# 02 - Analysis at full scale, from marts

Purpose: answer the operational questions on the whole dataset.

Everything here reads **marts**, not the 48M-row clean layer - the same
discipline the dashboard follows. If a question cannot be answered from a
mart, that is a signal the mart set is wrong, not a reason to scan raw.


In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from taxi.config import load_settings
from taxi.contract import Contract
from taxi.core import duck
from taxi.core.months import Month

settings = load_settings(ROOT / 'configs' / 'settings.yml')
contract = Contract.load(settings.contract_path)
print('contract', contract.version, contract.fingerprint)


contract 1.0.0 1.0.0+be8b35e8b06fd716


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

def mart(name, hive=True):
    directory = settings.paths.mart_dir(name)
    if not duck.table_exists(directory):
        raise FileNotFoundError(f'{name} not built - run: taxi marts build')
    with duck.connect(settings) as con:
        return con.execute(
            f'SELECT * FROM {duck.scan(settings.paths.glob(directory), hive=hive)}'
        ).df()

daily = mart('mart_daily_revenue')
print(f'{len(daily)} days, {daily.trip_count.sum():,} trips')


22:37:47 INFO    numexpr.utils          NumExpr defaulting to 16 threads.


FileNotFoundError: mart_daily_revenue not built - run: taxi marts build

## Q1 - When is demand highest?


In [ ]:
hourly = mart('mart_hourly_demand')
by_hour = hourly.groupby('hour_of_day').agg(
    trips=('trip_count', 'sum'), fare=('total_fare', 'sum')).reset_index()
by_hour['avg_fare'] = by_hour.fare / by_hour.trips

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(by_hour.hour_of_day, by_hour.trips)
axes[0].set_title('Trips by hour of day'); axes[0].set_xlabel('hour')
axes[1].plot(by_hour.hour_of_day, by_hour.avg_fare, marker='o')
axes[1].set_title('Average fare by hour'); axes[1].set_xlabel('hour')
fig.tight_layout()
by_hour.sort_values('trips', ascending=False).head(5)


Weekday and weekend are different products. Averaging them hides both.


In [ ]:
split = hourly.groupby(['hour_of_day', 'is_weekend'])['trip_count'].sum().unstack()
split.columns = ['weekday', 'weekend']
(split / split.sum()).plot(figsize=(12, 4), title='Share of trips by hour')


## Q2 - Which zones generate trips and revenue?


In [ ]:
zones = mart('dim_zone', hive=False).rename(columns={'location_id': 'PULocationID'})
pickup = mart('mart_zone_pickup_demand').groupby('PULocationID', as_index=False).agg(
    trips=('trip_count', 'sum'), revenue=('total_revenue', 'sum'),
    fare=('total_fare', 'sum'))
pickup = pickup.merge(zones, on='PULocationID', how='left')
pickup['avg_fare'] = pickup.fare / pickup.trips
top = pickup.nlargest(15, 'trips')
top[['zone', 'borough', 'trips', 'revenue', 'avg_fare']].round(2)


Volume and value are different rankings. Airport zones invert them.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pickup.trips, pickup.avg_fare, s=12, alpha=0.6)
for _, r in pickup.nlargest(8, 'revenue').iterrows():
    ax.annotate(str(r.get('zone', r.PULocationID))[:18], (r.trips, r.avg_fare),
                fontsize=8)
ax.set_xscale('log'); ax.set_xlabel('trips (log)'); ax.set_ylabel('average fare')
ax.set_title('Volume vs value by pickup zone')


## Q3 - How do fare, distance, duration and tipping relate?


In [ ]:
profile = mart('mart_fare_distance_profile').groupby('distance_bucket', as_index=False).agg(
    trips=('trip_count', 'sum'), fare=('total_fare', 'sum'),
    miles=('total_distance_miles', 'sum'), minutes=('total_duration_minutes', 'sum'))
profile['fare_per_mile'] = profile.fare / profile.miles
profile['avg_fare'] = profile.fare / profile.trips
profile['avg_minutes'] = profile.minutes / profile.trips
profile.round(2)


Fare per mile falls with distance because the flag-drop charge is spread
over more miles. A pre-trip model has to learn that curvature; a linear
model on distance alone cannot.


In [ ]:
tips = mart('mart_payment_tip_behavior')
card = tips[tips.payment_type == 1].groupby('hour_of_day', as_index=False).agg(
    tips=('total_tips', 'sum'), fare=('total_fare', 'sum'), trips=('trip_count', 'sum'))
card['tip_rate'] = card.tips / card.fare
card.plot(x='hour_of_day', y='tip_rate', figsize=(12, 4),
          title='Card tip rate by hour (cash tips are not recorded)')


## Q4 - What is being rejected?


In [ ]:
anomalies = mart('mart_anomalies')
primary = anomalies[anomalies.is_primary_reason]
by_reason = primary.groupby('rejection_reason', as_index=False)['rejected_count'].sum()
by_reason = by_reason.sort_values('rejected_count', ascending=False)
by_reason.plot.barh(x='rejection_reason', y='rejected_count', figsize=(10, 6),
                    title='Rejections by primary reason', legend=False)
by_reason


## Q5 - How do demand and revenue move over time?


In [ ]:
monthly = daily.groupby(['year', 'month'], as_index=False).agg(
    trips=('trip_count', 'sum'), revenue=('total_revenue', 'sum'))
monthly['period'] = monthly.apply(lambda r: f"{int(r.year):04d}-{int(r.month):02d}", axis=1)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly.period, monthly.revenue)
ax.set_title('Revenue by month'); plt.xticks(rotation=45)
monthly.round(0)


## If a question needed a raw scan

Record it. Either the mart set has a gap worth a new registry entry, or
the question is genuinely ad hoc and belongs in a notebook rather than in
the dashboard.
